# 🚗 Build phase — Chuyên gia 2/3: RT-DETR-R50vd trên 10 lớp BDD100K

## Thành viên nhóm: chỉ cần 3 thao tác
1. Mở notebook được giao → **Copy & Edit**.
2. Nhìn thanh bên phải: *Accelerator = GPU T4 x2*, *Internet = On*, và mục *Input* có dataset **BDD100K** (`solesensei_bdd100k`). Nếu thiếu, notebook sẽ dừng ngay trong phút đầu kèm lời nhắc cụ thể, không tốn quota.
3. **Save Version → Save** (để mặc định *Save & Run All*). Có thể đóng trình duyệt; khoảng 8.5 giờ sau, kết quả nằm ở tab *Output*.

Không sửa gì trong notebook. `EXPERT_ID = 2` đã được điền sẵn.

## Notebook tự làm các bước sau
1. **Kiểm tra môi trường:** GPU, Internet, phiên bản `transformers`, và đủ dữ liệu.
2. **Chuẩn bị dữ liệu.** Giữ **mọi box của 10 lớp**:
   * **BDD train** (69.863 ảnh) → train: dùng toàn bộ.
   * **BDD val** (10.000 ảnh) → **chỉ để xem**: đường cong loss trong lúc train (trên 1.000 ảnh cố định) và báo cáo từng lớp sau khi train. Không chọn checkpoint, không chỉnh siêu tham số theo val, để val vẫn dùng được cho bài báo.
   * **BDD test** → không đọc.
3. **Đo tốc độ** trong 40 bước, rồi tự chọn số bước train cho vừa `TRAIN_HOURS` = 8 giờ (vừa giới hạn 12 giờ mỗi phiên và hạn mức GPU khoảng 30 giờ mỗi tuần của Kaggle).
4. **Train** với seed riêng của chuyên gia 2. Có chốt chặn: tự lưu và dừng nếu phiên chạy tới 11 giờ.
5. **Kiểm tra** trên toàn bộ val: recall và precision theo từng lớp, recall theo thời tiết, thời điểm trong ngày và loại cảnh.
6. **Thống kê tiềm ẩn** trên toàn bộ tập train (chia đều cho các GPU, khoảng 20–25 phút): `expert_2_centroids.pt` theo định dạng M29 (chuyên gia 1 có thêm bản `centroids.pt`) và `expert_2_latent_stats.pt` cho H2.

## Kết quả trong Output
`expert_2.pth`, `expert_2_final/` (định dạng HuggingFace), `expert_2_manifest.json` (seed, số epoch, sha256 trọng số), `expert_2_class_report.txt`, `expert_2_loss_curve.csv` (loss train và val theo bước), `data_stats.json`.


In [ ]:
# ==============================================================================
# CẤU HÌNH (đã điền sẵn cho chuyên gia 2; thường không cần sửa)
# ==============================================================================
EXPERT_ID = 2       # seed: 1 -> 42, 2 -> 142, 3 -> 242
TRAIN_HOURS = 8.0   # quỹ thời gian cho riêng bước train
NUM_WORKERS = 2     # đặt 0 nếu dataloader bị treo
RESUME_FROM = ""    # để trống; chỉ dùng khi tiếp tục từ checkpoint-* của một phiên trước

import os, time
os.environ.update(EXPERT_ID=str(EXPERT_ID), NUM_WORKERS=str(NUM_WORKERS), RESUME_FROM=RESUME_FROM,
                  NB_START=str(time.time()))
os.chdir("/kaggle/working")


In [ ]:
# ==============================================================================
# KIỂM TRA MÔI TRƯỜNG
# ==============================================================================
import subprocess, sys, urllib.request, torch
n_gpu = torch.cuda.device_count()
print(f"[*] PyTorch {torch.__version__} | số GPU: {n_gpu} | " + ", ".join(torch.cuda.get_device_name(i) for i in range(n_gpu)))
assert n_gpu >= 1, "Chưa bật GPU: Settings -> Accelerator -> GPU T4 x2."
if n_gpu < 2:
    print("[!] Chỉ có 1 GPU: vẫn chạy được (batch hiệu dụng giữ nguyên 16) nhưng số epoch đạt được sẽ ít hơn.")
try:
    urllib.request.urlopen("https://huggingface.co", timeout=15)
except Exception as e:
    raise RuntimeError("Không có Internet (cần để tải trọng số PekingU/rtdetr_r50vd): Settings -> Internet -> On") from e
import transformers
# Cố định đúng phiên bản đã kiểm chứng (bản vá loss trong train.py đã được kiểm trên 4.57 và 5.16.1),
# để cả 3 chuyên gia chạy trên cùng thư viện.
TRANSFORMERS_VERSION = "5.16.1"
if transformers.__version__ != TRANSFORMERS_VERSION:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", f"transformers=={TRANSFORMERS_VERSION}"])
    print(f"[*] Đã cài transformers {TRANSFORMERS_VERSION} (các script chạy ở tiến trình riêng nên dùng ngay bản này).")
else:
    print(f"[V] transformers {transformers.__version__}")


## Chuẩn bị dữ liệu

In [ ]:
%%writefile common.py
# ==============================================================================
# CẤU HÌNH DÙNG CHUNG (chuẩn bị dữ liệu, huấn luyện, kiểm tra)
#
# Cách dùng dữ liệu BDD100K ở build phase:
#   BDD train (69.863 ảnh có nhãn) -> "train" : huấn luyện detector, dùng TOÀN BỘ
#   BDD val   (10.000 ảnh có nhãn) -> "val"   : CHỈ ĐỂ XEM: đường cong loss trong lúc train và báo cáo
#                                               từng lớp sau khi train. Không chọn checkpoint, không chỉnh
#                                               siêu tham số theo val, để val vẫn dùng được cho bài báo.
#   BDD test (20.000 ảnh)          -> KHÔNG ĐỌC : giữ cho bài báo hoặc cho bước hiệu chuẩn sau này
# ==============================================================================
import os, json, glob, hashlib

BDD_CLASSES = ['pedestrian', 'rider', 'car', 'truck', 'bus', 'train',
               'motorcycle', 'bicycle', 'traffic light', 'traffic sign']
CLASS2ID = {n: i for i, n in enumerate(BDD_CLASSES)}
ID2CLASS = {i: n for i, n in enumerate(BDD_CLASSES)}
# Nhãn BDD100K bản 2018 dùng 'person', 'bike', 'motor'; bản 2020 dùng tên mới.
CATEGORY_NORMALIZER = {
    'pedestrian': 'pedestrian', 'person': 'pedestrian', 'rider': 'rider', 'car': 'car',
    'truck': 'truck', 'bus': 'bus', 'train': 'train', 'motorcycle': 'motorcycle', 'motor': 'motorcycle',
    'bicycle': 'bicycle', 'bike': 'bicycle', 'traffic light': 'traffic light', 'traffic sign': 'traffic sign',
}

# Seed riêng của từng chuyên gia. Phải truyền vào TrainingArguments(seed=..., data_seed=...),
# vì Trainer gọi lại set_seed(args.seed) khi khởi tạo và sẽ ghi đè mọi seed đặt trước đó.
SEEDS = {1: 42, 2: 142, 3: 242}

WORK = os.environ.get("WORK_DIR", "/kaggle/working")
SPLITS = ("train", "val")
COMPACT = {s: os.path.join(WORK, f"bdd_det_{s}_compact.json") for s in SPLITS}
DATA_STATS = os.path.join(WORK, "data_stats.json")

# Một tập con cố định 1.000 ảnh val (chọn theo hash tên ảnh) để đo loss trong lúc train cho nhanh;
# báo cáo từng lớp sau khi train dùng toàn bộ 10.000 ảnh val.
def in_val_monitor(name):
    return int(hashlib.md5(name.encode()).hexdigest(), 16) % 10 == 0

# (Tùy chọn) danh sách ảnh train phải loại, một tên ảnh mỗi dòng (dùng cho bài báo, giao thức A2 của H1).
EXCLUDE_LIST_CANDIDATES = ["/kaggle/input/*/train_exclude.txt", "/kaggle/input/*/*/train_exclude.txt",
                           os.path.join(WORK, "train_exclude.txt")]

def find_exclude_list():
    for pat in EXCLUDE_LIST_CANDIDATES:
        hits = sorted(glob.glob(pat))
        if hits:
            return hits[0]
    return None

def find_bdd():
    """Tìm nhãn và ảnh của BDD train và BDD val trong /kaggle/input.
    Có thể chỉ định thẳng bằng biến môi trường TRAIN_JSON, TRAIN_IMG, VAL_JSON, VAL_IMG."""
    found = {k: os.environ.get(k) for k in ("TRAIN_JSON", "TRAIN_IMG", "VAL_JSON", "VAL_IMG")}
    if all(found.values()):
        return found
    for root, dirs, files in os.walk("/kaggle/input"):
        r = root.replace("\\", "/")
        for f in files:
            if f.endswith("labels_images_train.json"):
                found["TRAIN_JSON"] = found["TRAIN_JSON"] or os.path.join(root, f)
            if f.endswith("labels_images_val.json"):
                found["VAL_JSON"] = found["VAL_JSON"] or os.path.join(root, f)
        for split in ("train", "val"):
            key = f"{split.upper()}_IMG"
            # Bản Kaggle để một phần ảnh trong thư mục con (ví dụ test/trainA), nên chấp nhận thư mục
            # images/100k/<split> dù ảnh không nằm trực tiếp ở đó; image_map() quét đệ quy.
            if not found[key] and r.endswith(f"100k/{split}") and (
                    "images" in r or any(x.endswith(".jpg") for x in os.listdir(root)[:50])):
                found[key] = root
    return found

def raw_objects(item):
    """Hỗ trợ cả hai định dạng nhãn BDD: item['labels'] hoặc item['frames'][0]['objects']."""
    if item.get('labels') is not None:
        return item['labels']
    fr = item.get('frames') or []
    return fr[0].get('objects', []) if fr else []


In [ ]:
%%writefile prepare_data.py
# ==============================================================================
# CHUẨN BỊ DỮ LIỆU (tự bỏ qua nếu đã có kết quả)
#  - BDD train -> "train"; BDD val -> "val" (chỉ để xem). BDD test không đọc.
#  - Giữ TOÀN BỘ box của 10 lớp; chỉ bỏ box suy biến (rộng hoặc cao <= 0.5 px).
#  - Đếm box gốc và box giữ lại theo lớp; dừng lại nếu có box bị mất vì lý do khác.
#  - Tính hệ số lấy mẫu lặp (Repeat Factor Sampling; Gupta et al., 2019) cho lớp hiếm của tập train.
# ==============================================================================
import os, json, math, collections
from common import *

RFS_T = 0.02  # lớp xuất hiện trong dưới 2% số ảnh train sẽ được lặp lại

if all(os.path.exists(p) for p in list(COMPACT.values()) + [DATA_STATS]):
    print("[V] Dữ liệu đã được chuẩn bị từ trước, bỏ qua bước này.")
    raise SystemExit(0)

P = find_bdd()
for k, v in P.items():
    print(f"[*] {k:10s}: {v}")
missing = [k for k, v in P.items() if not v]
assert not missing, (
    f"Thiếu {missing}. Cần Add Input bộ BDD100K (ví dụ solesensei_bdd100k) có "
    "'bdd100k_labels_images_train.json', 'bdd100k_labels_images_val.json', 'images/100k/train', 'images/100k/val'.")

def image_map(d):
    m = {}
    for r, _, files in os.walk(d):
        for f in files:
            if f.endswith(".jpg"):
                m[f] = os.path.join(r, f)
    return m

excl_path = find_exclude_list()
exclude = set()
if excl_path:
    with open(excl_path, encoding="utf-8") as f:
        exclude = {ln.strip() if ln.strip().endswith(".jpg") else ln.strip() + ".jpg" for ln in f if ln.strip()}
    print(f"[V] Danh sách loại trừ cho tập train: {excl_path} ({len(exclude)} ảnh)")

raw_cnt = collections.Counter(); degen_cnt = collections.Counter(); other_cats = collections.Counter()
kept = {s: collections.Counter() for s in SPLITS}
out = {s: [] for s in SPLITS}
info = collections.Counter(); max_boxes = 0
for source, json_key, img_key in (("bdd_train", "TRAIN_JSON", "TRAIN_IMG"), ("bdd_val", "VAL_JSON", "VAL_IMG")):
    img_map = image_map(P[img_key])
    with open(P[json_key], encoding="utf-8") as f:
        raw = json.load(f)
    labeled = set()
    for item in raw:
        name = item['name'] if item['name'].endswith(".jpg") else item['name'] + ".jpg"
        labeled.add(name)
        path = img_map.get(name)
        if path is None:
            info[f"{source}: nhãn không có ảnh"] += 1; continue
        if source == "bdd_train" and name in exclude:
            info["bdd_train: loại trừ"] += 1; continue
        split = "train" if source == "bdd_train" else "val"
        boxes = []
        for o in raw_objects(item):
            if 'box2d' not in o:
                continue
            cat = CATEGORY_NORMALIZER.get(o.get('category'))
            if cat is None:
                other_cats[o.get('category')] += 1; continue
            raw_cnt[cat] += 1
            b = o['box2d']; x1, y1, x2, y2 = float(b['x1']), float(b['y1']), float(b['x2']), float(b['y2'])
            if x2 - x1 > 0.5 and y2 - y1 > 0.5:
                boxes.append([x1, y1, x2, y2, CLASS2ID[cat]]); kept[split][cat] += 1
            else:
                degen_cnt[cat] += 1
        info[f"{source}: ảnh không có box"] += not boxes
        max_boxes = max(max_boxes, len(boxes))
        out[split].append({"file": path, "name": name, "boxes": boxes, "attributes": item.get("attributes", {}),
                           "monitor": split == "val" and in_val_monitor(name)})
    # Ảnh có trong thư mục nhưng không có mục nào trong file nhãn: bỏ qua, vì mọi object trong ảnh sẽ bị coi là nền.
    unlabeled = sorted(set(img_map) - labeled)
    info[f"{source}: ảnh không có nhãn"] = len(unlabeled)
    with open(os.path.join(WORK, f"unlabeled_images_{source}.txt"), "w") as f:
        f.write("\n".join(unlabeled))
    del raw

img_with = collections.Counter()
for rec in out["train"]:
    img_with.update({BDD_CLASSES[b[4]] for b in rec["boxes"]})
n_train = len(out["train"])
r_c = {c: max(1.0, math.sqrt(RFS_T / (img_with[c] / n_train))) if img_with[c] else 1.0 for c in BDD_CLASSES}
for rec in out["train"]:
    rec["repeat"] = max([r_c[BDD_CLASSES[b[4]]] for b in rec["boxes"]], default=1.0)
for s in SPLITS:
    with open(COMPACT[s], "w") as f:
        json.dump(out[s], f)

kept_all = collections.Counter()
for s in SPLITS:
    kept_all.update(kept[s])
stats = dict(sources={k: v for k, v in P.items()}, split_rule="train = toàn bộ BDD train; val = BDD val, chỉ để xem "
             "(loss theo dõi trên tập con md5(tên ảnh) % 10 == 0, báo cáo từng lớp trên toàn bộ val); BDD test không đọc",
             n_val_monitor=sum(r["monitor"] for r in out["val"]),
             n_images={s: len(out[s]) for s in SPLITS}, info=dict(info), exclude_list=excl_path,
             train_filtered=bool(excl_path), max_boxes_per_image=max_boxes, raw_boxes=dict(raw_cnt),
             kept_boxes={s: dict(kept[s]) for s in SPLITS}, degenerate_boxes=dict(degen_cnt),
             train_images_with_class=dict(img_with), rfs_t=RFS_T, rfs_repeat_per_class=r_c,
             train_images_per_epoch=sum(r["repeat"] for r in out["train"]), box2d_of_other_categories=dict(other_cats))
with open(DATA_STATS, "w") as f:
    json.dump(stats, f, indent=2, ensure_ascii=False)

print("\n[*] Số ảnh: " + " | ".join(f"{s}: {len(out[s])}" for s in SPLITS))
for k, v in sorted(info.items()):
    print(f"    {k}: {v}")
print(f"\n{'lớp':14s} {'box gốc':>9s} {'train':>9s} {'val':>9s} {'suy biến':>9s} {'lặp RFS':>8s}")
for c in BDD_CLASSES:
    print(f"{c:14s} {raw_cnt[c]:9d} {kept['train'][c]:9d} {kept['val'][c]:9d} {degen_cnt[c]:9d} {r_c[c]:8.2f}")
assert all(raw_cnt[c] == kept_all[c] + degen_cnt[c] for c in BDD_CLASSES), "Có box bị mất ngoài lý do suy biến!"
print(f"[V] Mọi box của 10 lớp đều được giữ (trừ box suy biến). Nhiều box nhất trong 1 ảnh: {max_boxes}. "
      f"Ảnh train/epoch sau RFS ≈ {round(stats['train_images_per_epoch'])}")
if max_boxes > 300:
    print("[!] Có ảnh nhiều hơn 300 object, vượt số query của RT-DETR (300).")


In [ ]:
!python prepare_data.py


## Huấn luyện

In [ ]:
%%writefile train.py
# ==============================================================================
# HUẤN LUYỆN MỘT CHUYÊN GIA RT-DETR-R50vd TRÊN 10 LỚP BDD100K (1 hoặc 2 GPU, DDP qua torchrun)
# Biến môi trường (đặt tự động ở notebook):
#   EXPERT_ID   : 1, 2 hoặc 3
#   BENCH=1     : chỉ chạy 40 bước để đo tốc độ, ghi bench_K.json, không lưu mô hình
#   MAX_STEPS   : số bước tối ưu hóa (tính từ tốc độ đo được và quỹ thời gian)
#   NB_START    : thời điểm notebook bắt đầu (epoch giây), dùng cho chốt chặn 11 giờ
# ==============================================================================
import os, json, math, random, hashlib, datetime, time
import numpy as np
import torch
import torch.distributed as dist
from torch.utils.data import Dataset
from PIL import Image, ImageEnhance
import transformers
from transformers import (RTDetrForObjectDetection, RTDetrImageProcessor, TrainingArguments, Trainer,
                          TrainerCallback)
from common import *

EXPERT_ID = int(os.environ["EXPERT_ID"])
SEED = SEEDS[EXPERT_ID]
BENCH = os.environ.get("BENCH") == "1"
MAX_STEPS = int(os.environ.get("MAX_STEPS", 0))
NB_START = float(os.environ.get("NB_START", time.time()))
HARD_STOP_H = float(os.environ.get("HARD_STOP_H", 11.0))  # Kaggle dừng phiên ở 12 giờ
LR, LR_BACKBONE = 1e-4, 1e-5             # cấu hình fine-tune thường dùng cho RT-DETR
PER_GPU_BATCH, EFFECTIVE_BATCH = 4, 16
WORLD = int(os.environ.get("WORLD_SIZE", 1))
ACCUM = max(1, EFFECTIVE_BATCH // (PER_GPU_BATCH * WORLD))  # giữ batch hiệu dụng 16 với 1 hay 2 GPU
NUM_WORKERS = int(os.environ.get("NUM_WORKERS", 2))
RESUME_FROM = os.environ.get("RESUME_FROM") or None
OUT_DIR = os.path.join(WORK, f"expert_{EXPERT_ID}" if not BENCH else f"bench_tmp_{EXPERT_ID}")
transformers.set_seed(SEED)

# ------------------------------------------------------------------------------
# SỬA LỖI CỦA TRANSFORMERS (có ở 4.57 và 5.16): trong RTDetrForObjectDetectionLoss, query khử nhiễu (denoising)
# chỉ được tách khỏi các lớp decoder phụ; loss của lớp decoder CUỐI vẫn dùng logits/pred_boxes gồm cả
# query khử nhiễu (300 + 198 = 498 query). Bộ ghép Hungarian vì thế ghép GT vào query khử nhiễu (vốn nằm sát GT),
# 300 query thật của lớp cuối chỉ nhận mục tiêu "nền", và điểm phân loại khi suy luận bị ép về ~0.
# (Đây là nguyên nhân chuyên gia 2 lần đầu có 0 dự đoán trên val.) Code gốc RT-DETR tách trước khi ghép.
# ------------------------------------------------------------------------------
import transformers.loss.loss_utils as _hf_loss_utils
_HF_RTDETR_LOSS = _hf_loss_utils.LOSS_MAPPING["RTDetrForObjectDetection"]
_LOSS_FIX_SEEN = {"n": 0}

def rtdetr_loss_fixed(logits, labels, device, pred_boxes, config, outputs_class=None, outputs_coord=None,
                      *args, denoising_meta_values=None, **kwargs):
    if denoising_meta_values is not None:
        n_dn = denoising_meta_values["dn_num_split"][0]  # thứ tự: [query khử nhiễu, query thật]
        logits, pred_boxes = logits[:, n_dn:], pred_boxes[:, n_dn:]
        if _LOSS_FIX_SEEN["n"] == 0:
            assert logits.shape[1] == config.num_queries, (logits.shape, config.num_queries)
            print(f"[V] Đã tách {n_dn} query khử nhiễu khỏi loss của lớp decoder cuối "
                  f"(còn {logits.shape[1]} query thật).", flush=True)
        _LOSS_FIX_SEEN["n"] += 1
    return _HF_RTDETR_LOSS(logits, labels, device, pred_boxes, config, outputs_class, outputs_coord,
                           *args, denoising_meta_values=denoising_meta_values, **kwargs)

_hf_loss_utils.LOSS_MAPPING["RTDetrForObjectDetection"] = rtdetr_loss_fixed


class BDDDet(Dataset):
    """Augmentation ngẫu nhiên (lật ngang, độ sáng/tương phản/bão hòa) vừa tăng độ bền vừa là nguồn
    đa dạng giữa các chuyên gia; nó không loại bỏ object nào."""
    def __init__(self, items, processor, augment):
        self.items, self.processor, self.augment = items, processor, augment

    def __len__(self):
        return len(self.items)

    def __getitem__(self, idx):
        rec = self.items[idx]
        image = Image.open(rec["file"]).convert("RGB")
        W, _ = image.size
        boxes = [list(b) for b in rec["boxes"]]
        if self.augment:
            if random.random() < 0.5:
                image = image.transpose(Image.FLIP_LEFT_RIGHT)
                boxes = [[W - x2, y1, W - x1, y2, c] for x1, y1, x2, y2, c in boxes]
            for enh in (ImageEnhance.Brightness, ImageEnhance.Contrast, ImageEnhance.Color):
                image = enh(image).enhance(random.uniform(0.8, 1.2))
        anns = [{"bbox": [x1, y1, x2 - x1, y2 - y1], "category_id": int(c), "area": (x2 - x1) * (y2 - y1), "iscrowd": 0}
                for x1, y1, x2, y2, c in boxes]
        enc = self.processor(images=image, annotations={"image_id": idx, "annotations": anns}, return_tensors="pt")
        return {"pixel_values": enc["pixel_values"][0], "labels": enc["labels"][0]}


def collate_fn(batch):
    return {"pixel_values": torch.stack([b["pixel_values"] for b in batch]), "labels": [b["labels"] for b in batch]}


def expand_rfs(items, seed):
    """Lặp ảnh theo hệ số RFS với làm tròn ngẫu nhiên (cố định theo seed của chuyên gia)."""
    rng = np.random.default_rng(seed)
    out = []
    for rec in items:
        r = rec.get("repeat", 1.0)
        out.extend([rec] * (int(math.floor(r)) + int(rng.random() < r - math.floor(r))))
    return out


def build_model():
    model = RTDetrForObjectDetection.from_pretrained(
        "PekingU/rtdetr_r50vd", id2label=ID2CLASS, label2id=CLASS2ID,
        num_labels=len(BDD_CLASSES), ignore_mismatched_sizes=True)
    # Khởi tạo hàng của đầu phân loại từ lớp COCO tương ứng để hội tụ nhanh hơn.
    # rider và traffic sign không có lớp COCO tương ứng nên giữ khởi tạo mặc định.
    # Config của PekingU dùng 'motorbike' thay cho 'motorcycle', nên mỗi lớp có danh sách tên thay thế.
    pairs = {'pedestrian': ['person'], 'car': ['car'], 'truck': ['truck'], 'bus': ['bus'], 'train': ['train'],
             'motorcycle': ['motorcycle', 'motorbike'], 'bicycle': ['bicycle'], 'traffic light': ['traffic light']}
    try:
        coco = RTDetrForObjectDetection.from_pretrained("PekingU/rtdetr_r50vd")
        c2i = {v: int(k) for k, v in coco.config.id2label.items()}
        done = []
        with torch.no_grad():
            for bdd, names in pairs.items():
                j = next((c2i[n] for n in names if n in c2i), None)
                if j is None:
                    continue
                i = CLASS2ID[bdd]
                for l in range(len(model.model.decoder.class_embed)):
                    model.model.decoder.class_embed[l].weight[i] = coco.model.decoder.class_embed[l].weight[j]
                    model.model.decoder.class_embed[l].bias[i] = coco.model.decoder.class_embed[l].bias[j]
                model.model.enc_score_head.weight[i] = coco.model.enc_score_head.weight[j]
                model.model.enc_score_head.bias[i] = coco.model.enc_score_head.bias[j]
                done.append(bdd)
        del coco
        print(f"[V] Đã khởi tạo đầu phân loại từ COCO cho {len(done)}/10 lớp: {done}")
    except Exception as e:  # không chặn việc train
        print(f"[!] Không khởi tạo được từ COCO ({e}); dùng khởi tạo mặc định.")
    return model


class TimeGuard(TrainerCallback):
    """Dừng an toàn (lưu checkpoint rồi thoát) khi notebook chạy gần tới giới hạn 12 giờ của Kaggle.
    Mọi tiến trình DDP thống nhất quyết định qua all_reduce để không bị treo."""
    def on_step_end(self, args, state, control, **kw):
        if state.global_step % 50 and state.global_step != 1:
            return
        flag = torch.tensor([float(time.time() - NB_START > HARD_STOP_H * 3600)],
                            device="cuda" if torch.cuda.is_available() else "cpu")
        if dist.is_available() and dist.is_initialized():
            dist.all_reduce(flag, op=dist.ReduceOp.MAX)
        if flag.item() > 0:
            print(f"[!] Gần hết thời gian phiên Kaggle: dừng ở bước {state.global_step} và lưu checkpoint.")
            control.should_save = True; control.should_training_stop = True


class Speed(TrainerCallback):
    """Đo tốc độ (bước tối ưu hóa mỗi giây) từ bước 10 đến bước cuối, dùng cho chế độ BENCH."""
    def __init__(self): self.t = {}
    def on_step_end(self, args, state, control, **kw):
        self.t[state.global_step] = time.time()


class NanGuard(TrainerCallback):
    """Dừng ngay nếu loss được ghi log không hữu hạn (NaN/Inf). Loss log đã được gộp giữa các GPU,
    nên mọi tiến trình nhận cùng giá trị và dừng cùng lúc."""
    def on_log(self, args, state, control, logs=None, **kw):
        v = (logs or {}).get("loss")
        if v is not None and not math.isfinite(v):
            print(f"[X] Loss = {v} ở bước {state.global_step}: dừng train.", flush=True)
            control.should_training_stop = True


PROBE_EVERY = int(os.environ.get("PROBE_EVERY", 2000))
PROBE_MIN_STEP = int(os.environ.get("PROBE_MIN_STEP", 4000))


class ScoreProbe(TrainerCallback):
    """Kiểm tra sớm: mỗi PROBE_EVERY bước, chạy suy luận trên 32 ảnh TRAIN cố định (không dùng val) và ghi
    điểm tin cậy cao nhất trung bình + số box > 0.3 vào expert_K_probe.csv. Một mô hình đang học bình thường
    giữ điểm này cao (ban đầu khoảng 0.85). Từ bước PROBE_MIN_STEP, nếu điểm < 0.1 thì mô hình đang suy biến
    (như lỗi loss đã sửa ở trên): dừng ngay để không tốn quota. Mọi GPU chạy cùng phép đo trên cùng trọng số
    nên đưa ra cùng quyết định, không cần trao đổi."""
    def __init__(self, items, processor):
        self.proc = processor
        self.ims = [Image.open(r["file"]).convert("RGB") for r in items[:32]]
        self.path = os.path.join(WORK, f"expert_{EXPERT_ID}_probe.csv")

    def on_step_end(self, args, state, control, model=None, **kw):
        if state.global_step % PROBE_EVERY:
            return
        m = model.module if hasattr(model, "module") else model
        was_training = m.training
        m.eval()
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16, enabled=torch.cuda.is_available()):
            inp = self.proc(images=self.ims, return_tensors="pt").to(next(m.parameters()).device)
            out = m(pixel_values=inp["pixel_values"])
        s = out.logits.float().sigmoid().amax(dim=-1)                 # [ảnh, query]
        mean_max = s.amax(dim=1).mean().item(); n_box = int((s > 0.3).sum())
        if was_training:
            m.train()
        if state.is_world_process_zero:
            new = not os.path.exists(self.path)
            with open(self.path, "a") as f:
                f.write(("step,mean_max_score,boxes_over_0.3\n" if new else "") + f"{state.global_step},{mean_max:.4f},{n_box}\n")
            print(f"[probe] bước {state.global_step}: điểm cao nhất TB {mean_max:.3f}, box > 0.3: {n_box}", flush=True)
        if state.global_step >= PROBE_MIN_STEP and (not math.isfinite(mean_max) or mean_max < 0.1):
            if state.is_world_process_zero:
                print(f"[X] Điểm tin cậy suy biến ({mean_max:.3f} < 0.1): dừng train. Xem expert_{EXPERT_ID}_probe.csv.", flush=True)
            control.should_training_stop = True


def weights_sha256(sd):
    h = hashlib.sha256()
    for k in sorted(sd):
        h.update(k.encode()); h.update(sd[k].detach().cpu().contiguous().numpy().tobytes())
    return h.hexdigest()


if __name__ == "__main__":
    with open(COMPACT["train"]) as f:
        items = json.load(f)
    with open(DATA_STATS) as f:
        stats = json.load(f)
    train_items = expand_rfs(items, SEED)
    with open(COMPACT["val"]) as f:
        val_monitor = [r for r in json.load(f) if r["monitor"]]  # 1.000 ảnh val cố định, CHỈ để vẽ đường cong loss
    processor = RTDetrImageProcessor.from_pretrained("PekingU/rtdetr_r50vd")
    model = build_model()

    backbone = [p for n, p in model.named_parameters() if "backbone" in n and p.requires_grad]
    rest = [p for n, p in model.named_parameters() if "backbone" not in n and p.requires_grad]
    optimizer = torch.optim.AdamW([{"params": backbone, "lr": LR_BACKBONE}, {"params": rest, "lr": LR}],
                                  lr=LR, weight_decay=1e-4)
    steps = 40 if BENCH else MAX_STEPS
    assert steps > 0, "Thiếu MAX_STEPS: hãy chạy cell đo tốc độ trước."
    args = TrainingArguments(
        output_dir=OUT_DIR,
        seed=SEED, data_seed=SEED,            # BẮT BUỘC: nếu thiếu, Trainer đặt lại seed = 42 cho mọi chuyên gia
        max_steps=steps,
        per_device_train_batch_size=PER_GPU_BATCH, gradient_accumulation_steps=ACCUM,
        learning_rate=LR, lr_scheduler_type="cosine", warmup_steps=min(1000, max(1, steps // 20)),
        max_grad_norm=0.1, fp16=torch.cuda.is_available(),
        # Val CHỈ ĐỂ XEM: đo loss trên 1.000 ảnh val mỗi 2.000 bước để thấy overfit. Không chọn checkpoint theo
        # val (load_best_model_at_end=False) và không có early stopping: mô hình lưu ra luôn là bản cuối cùng.
        eval_strategy="no" if BENCH else "steps", eval_steps=int(os.environ.get("EVAL_STEPS", 2000)), prediction_loss_only=True,
        per_device_eval_batch_size=PER_GPU_BATCH, load_best_model_at_end=False,
        save_strategy="no" if BENCH else "steps", save_steps=2000, save_total_limit=2,
        logging_steps=50, report_to="none", disable_tqdm=False,
        dataloader_num_workers=NUM_WORKERS, dataloader_persistent_workers=NUM_WORKERS > 0,
        ddp_find_unused_parameters=False, remove_unused_columns=False,
    )
    speed = Speed()
    trainer = Trainer(model=model, args=args, train_dataset=BDDDet(train_items, processor, augment=True),
                      eval_dataset=None if BENCH else BDDDet(val_monitor, processor, augment=False),
                      data_collator=collate_fn, optimizers=(optimizer, None),
                      callbacks=[speed] if BENCH else [TimeGuard(), NanGuard(), ScoreProbe(items, processor)])
    eff_batch = PER_GPU_BATCH * ACCUM * WORLD
    if trainer.is_world_process_zero():
        print("=" * 70)
        print(f"[*] CHUYÊN GIA {EXPERT_ID} | seed={SEED} | GPU={WORLD} | batch hiệu dụng={eff_batch} | bước={steps}"
              f" | ≈{steps * eff_batch / len(train_items):.2f} epoch | {'ĐO TỐC ĐỘ' if BENCH else 'TRAIN'}")
        print("=" * 70)

    trainer.train(resume_from_checkpoint=None if BENCH else RESUME_FROM)

    if trainer.is_world_process_zero():
        if BENCH:
            t = speed.t; a, b = 10, max(t)
            sps = (b - a) / (t[b] - t[a])
            with open(os.path.join(WORK, f"bench_{EXPERT_ID}.json"), "w") as f:
                json.dump({"steps_per_sec": sps, "effective_batch": eff_batch, "world": WORLD,
                           "images_per_epoch": len(train_items)}, f)
            print(f"[V] Tốc độ: {sps:.3f} bước/giây ({sps * eff_batch:.1f} ảnh/giây)")
        else:
            final_dir = os.path.join(WORK, f"expert_{EXPERT_ID}_final")
            trainer.save_model(final_dir); processor.save_pretrained(final_dir)
            sd = trainer.model.module.state_dict() if hasattr(trainer.model, "module") else trainer.model.state_dict()
            pth = os.path.join(WORK, f"expert_{EXPERT_ID}.pth")
            torch.save(sd, pth)
            manifest = dict(expert_id=EXPERT_ID, seed=SEED, data_seed=SEED, max_steps=steps,
                            steps_done=trainer.state.global_step,
                            epochs_done=trainer.state.global_step * eff_batch / len(train_items),
                            lr=LR, lr_backbone=LR_BACKBONE, effective_batch=eff_batch, world=WORLD,
                            classes=BDD_CLASSES, rfs_t=stats["rfs_t"], images_per_epoch=len(train_items),
                            split_rule=stats["split_rule"], val_usage="chỉ để xem: không chọn checkpoint, không chỉnh siêu tham số",
                            rtdetr_loss_fix="tách query khử nhiễu khỏi loss của lớp decoder cuối (lỗi transformers)",
                            stopped_early=trainer.state.global_step < steps,
                            n_images=stats["n_images"], train_filtered=stats["train_filtered"],
                            kept_boxes_train=stats["kept_boxes"]["train"], weights_sha256=weights_sha256(sd),
                            last_log=trainer.state.log_history[-1],
                            transformers=transformers.__version__, torch=torch.__version__,
                            finished_utc=datetime.datetime.utcnow().isoformat())
            with open(os.path.join(WORK, f"expert_{EXPERT_ID}_manifest.json"), "w") as f:
                json.dump(manifest, f, indent=2)
            # Đường cong loss: train (mỗi 50 bước) và val (mỗi 2.000 bước), chỉ để xem
            with open(os.path.join(WORK, f"expert_{EXPERT_ID}_loss_curve.csv"), "w") as f:
                f.write("step,train_loss,val_loss\n")
                for h in trainer.state.log_history:
                    if "loss" in h or "eval_loss" in h:
                        f.write(f"{h.get('step')},{h.get('loss', '')},{h.get('eval_loss', '')}\n")
            print(f"[V] Đã lưu {pth}, {final_dir}/ và expert_{EXPERT_ID}_manifest.json")
            print(f"    {manifest['epochs_done']:.2f} epoch | sha256 trọng số: {manifest['weights_sha256'][:16]}…")


In [ ]:
# ==============================================================================
# ĐO TỐC ĐỘ (40 bước) VÀ CHỌN SỐ BƯỚC TRAIN CHO VỪA QUỸ THỜI GIAN
# ==============================================================================
import json, shutil
!BENCH=1 torchrun --nproc_per_node={n_gpu} train.py
b = json.load(open(f"/kaggle/working/bench_{EXPERT_ID}.json"))
shutil.rmtree(f"/kaggle/working/bench_tmp_{EXPERT_ID}", ignore_errors=True)
MAX_STEPS = int(TRAIN_HOURS * 3600 * b["steps_per_sec"] * 0.90)   # chừa 10% cho lưu checkpoint và đo loss val
epochs = MAX_STEPS * b["effective_batch"] / b["images_per_epoch"]
os.environ["MAX_STEPS"] = str(MAX_STEPS)
print(f"[*] {b['steps_per_sec'] * b['effective_batch']:.1f} ảnh/giây -> {MAX_STEPS} bước ≈ {epochs:.1f} epoch trong {TRAIN_HOURS} giờ")
if epochs < 3:
    print("[!] Dưới 3 epoch: mô hình sẽ chưa hội tụ tốt. Hãy kiểm tra đã chọn GPU T4 x2 chưa.")


In [ ]:
# TRAIN (đọc MAX_STEPS từ cell trên)
!torchrun --nproc_per_node={n_gpu} train.py


## Kiểm tra từng lớp và edge case

In [ ]:
%%writefile check_classes.py
# ==============================================================================
# KIỂM TRA SAU KHI TRAIN trên toàn bộ BDD val (CHỈ ĐỂ XEM, không dùng để chọn hay chỉnh gì):
#  (1) recall/precision theo từng lớp -> mô hình đã học đủ 10 lớp chưa
#  (2) recall theo nhóm điều kiện (thời tiết, thời điểm, loại cảnh) -> edge case
#  (3) so sánh với các chuyên gia khác (nếu có trong Input) -> có phải bản sao không
# Kết quả ghi vào expert_K_class_report.txt.
# ==============================================================================
import os, sys, json, glob, itertools, collections
import numpy as np, torch
from PIL import Image
from transformers import RTDetrForObjectDetection, RTDetrImageProcessor
from common import *

class Tee:  # in ra màn hình đồng thời ghi vào file báo cáo
    def __init__(self, path): self.f = open(path, "w", encoding="utf-8"); self.o = sys.stdout
    def write(self, s): self.o.write(s); self.f.write(s)
    def flush(self): self.o.flush(); self.f.flush()

EXPERT_ID = int(sys.argv[1]); N_IMG = int(sys.argv[2]) if len(sys.argv) > 2 else 10**9
sys.stdout = Tee(os.path.join(WORK, f"expert_{EXPERT_ID}_class_report.txt"))
dev = "cuda" if torch.cuda.is_available() else "cpu"
final_dir = os.path.join(WORK, f"expert_{EXPERT_ID}_final")
model = RTDetrForObjectDetection.from_pretrained(final_dir).to(dev).eval()
proc = RTDetrImageProcessor.from_pretrained(final_dir)
with open(COMPACT["val"]) as f:
    items = json.load(f)[:N_IMG]

def iou(a, b):
    x1 = np.maximum(a[:, None, 0], b[None, :, 0]); y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2]); y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    ar = lambda z: (z[:, 2] - z[:, 0]) * (z[:, 3] - z[:, 1])
    return inter / (ar(a)[:, None] + ar(b)[None, :] - inter + 1e-9)

gt = np.zeros(10, int); tp = np.zeros(10, int); npred = np.zeros(10, int); pred_tp = np.zeros(10, int)
strata = collections.defaultdict(lambda: [0, 0, 0])  # (thuộc tính, giá trị) -> [số ảnh, GT, TP]
for k in range(0, len(items), 16):
    batch = items[k:k + 16]
    ims = [Image.open(r["file"]).convert("RGB") for r in batch]
    inp = proc(images=ims, return_tensors="pt").to(dev)
    with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16, enabled=dev == "cuda"):
        out = model(**inp)
    out["logits"] = out.logits.float(); out["pred_boxes"] = out.pred_boxes.float()
    res = proc.post_process_object_detection(out, threshold=0.3, target_sizes=[im.size[::-1] for im in ims])
    for r, p in zip(batch, res):
        g = np.array(r["boxes"], float).reshape(-1, 5)
        pb, pl = p["boxes"].cpu().numpy(), p["labels"].cpu().numpy()
        img_gt = img_tp = 0
        for c in range(10):
            gc, pc = g[g[:, 4] == c, :4], pb[pl == c]
            gt[c] += len(gc); npred[c] += len(pc); img_gt += len(gc)
            if len(gc) and len(pc):
                m = iou(gc, pc) >= 0.5
                t = int(m.any(1).sum()); tp[c] += t; img_tp += t; pred_tp[c] += int(m.any(0).sum())
        for a in ("weather", "timeofday", "scene"):
            s = strata[(a, r.get("attributes", {}).get(a, "?"))]
            s[0] += 1; s[1] += img_gt; s[2] += img_tp

print(f"[*] Chuyên gia {EXPERT_ID}: {len(items)} ảnh BDD val (không dùng khi train; chỉ để xem), ngưỡng 0.3, IoU 0.5")
print(f"\n{'lớp':14s} {'GT':>7s} {'dự đoán':>8s} {'recall':>7s} {'precision':>9s}")
weak = []
for c, n in enumerate(BDD_CLASSES):
    rec = tp[c] / gt[c] if gt[c] else float("nan"); prec = pred_tp[c] / npred[c] if npred[c] else float("nan")
    note = "  (dưới 20 GT: không đủ tin cậy)" if gt[c] < 20 else ""
    print(f"{n:14s} {gt[c]:7d} {npred[c]:8d} {rec:7.3f} {prec:9.3f}{note}")
    if gt[c] >= 20 and rec < 0.05:
        weak.append(n)
print("\n[!] CHƯA HỌC ĐƯỢC:" if weak else "\n[V] Mọi lớp đủ dữ liệu kiểm tra đều đã được học.", weak or "")

print(f"\n{'nhóm điều kiện':32s} {'ảnh':>6s} {'GT':>7s} {'recall':>7s}")
for (a, v), (n_img, g_, t_) in sorted(strata.items(), key=lambda kv: (kv[0][0], -kv[1][0])):
    note = "  (dưới 30 ảnh: không đủ tin cậy)" if n_img < 30 else ""
    print(f"{a + ' = ' + str(v):32s} {n_img:6d} {g_:7d} {t_ / g_ if g_ else float('nan'):7.3f}{note}")

paths = sorted(set(glob.glob(os.path.join(WORK, "expert_*.pth")) + glob.glob("/kaggle/input/**/expert_*.pth", recursive=True)))
heads = {}
for p in paths:
    sd = torch.load(p, map_location="cpu", weights_only=True)
    heads[p] = torch.cat([sd[k].flatten().float() for k in sorted(sd) if "class_embed" in k or "enc_score_head" in k])
for a, b in itertools.combinations(heads, 2):
    d = (heads[a] - heads[b]).norm() / heads[a].norm()
    flag = "  <-- GIỐNG HỆT, không phải ensemble!" if d < 1e-6 else ""
    print(f"[*] Lệch tương đối đầu phân loại {os.path.basename(a)} vs {os.path.basename(b)}: {d:.4f}{flag}")


In [ ]:
!python check_classes.py {EXPERT_ID}


## Thống kê không gian tiềm ẩn (centroids.pt cho M29, latent_stats.pt cho H2)

In [ ]:
%%writefile latent_stats.py
# ==============================================================================
# THỐNG KÊ KHÔNG GIAN TIỀM ẨN (chạy sau khi train, cho MỌI chuyên gia, trên TOÀN BỘ BDD train)
#  Theo M29 Mục 4.B, Max-IoU Assignment: với mỗi GT, chọn query có IoU >= 0.5 cao nhất VÀ đoán đúng lớp;
#  lấy vector ẩn 256D của lớp decoder cuối (decoder_hidden_states[-1]).
#  Hai bước:
#   python latent_stats.py extract <K> <phần> <số phần>  : trích vector trên một phần tập train (mỗi GPU một phần)
#   python latent_stats.py merge   <K> <số phần>          : gộp các phần và tính thống kê
#  Đầu ra:
#   - expert_K_centroids.pt  : ĐÚNG định dạng M29 (pca, centroids, inv_covs, class_names, d_in, d_out, jitter_eps),
#                              PCA 256 -> 64, Σ_k^-1 theo lớp (+1e-6 I). Chuyên gia 1 chép thêm thành centroids.pt.
#                              Lớp có dưới 192 vector dùng hiệp phương sai co rút Ledoit-Wolf ("shrunk_classes").
#   - expert_K_latent_stats.pt: chỉ chứa tensor (nạp được bằng weights_only=True), cho tín hiệu FCD của H2:
#                              tâm lớp 256D, precision của hiệp phương sai gộp Ledoit-Wolf 256D, trung vị/MAD của
#                              khoảng cách Mahalanobis trên các vector TP (để chuẩn hóa robust).
# Không dùng val: thống kê tham chiếu chỉ lấy từ train.
# ==============================================================================
import os, sys, json, collections
import numpy as np, torch
from common import *

MODE, EXPERT_ID = sys.argv[1], int(sys.argv[2])
CAP_PER_CLASS = 50000           # giới hạn mỗi lớp (car, traffic sign, traffic light vượt mức này); lớp hiếm lấy hết
JITTER = 1e-6
part_path = lambda i: os.path.join(WORK, f"expert_{EXPERT_ID}_latent_part{i}.npz")

if MODE == "extract":
    from PIL import Image
    from torchvision.ops import box_iou
    from transformers import RTDetrForObjectDetection, RTDetrImageProcessor
    part, nparts = int(sys.argv[3]), int(sys.argv[4])
    dev = f"cuda:{part % max(1, torch.cuda.device_count())}" if torch.cuda.is_available() else "cpu"
    final_dir = os.path.join(WORK, f"expert_{EXPERT_ID}_final")
    model = RTDetrForObjectDetection.from_pretrained(final_dir).to(dev).eval()
    proc = RTDetrImageProcessor.from_pretrained(final_dir)
    with open(COMPACT["train"]) as f:
        items = [r for i, r in enumerate(json.load(f)) if r["boxes"] and i % nparts == part]
    print(f"[*] Chuyên gia {EXPERT_ID}, phần {part + 1}/{nparts} trên {dev}: {len(items)} ảnh train", flush=True)
    cap = CAP_PER_CLASS // nparts
    emb = collections.defaultdict(list); seen = collections.Counter()
    rng = np.random.default_rng(1000 * EXPERT_ID + part); n_gt = n_kept = 0
    for k in range(0, len(items), 16):
        batch = items[k:k + 16]
        ims = [Image.open(r["file"]).convert("RGB") for r in batch]
        inp = proc(images=ims, return_tensors="pt").to(dev)
        with torch.no_grad(), torch.autocast(device_type="cuda", dtype=torch.float16, enabled=dev != "cpu"):
            out = model(pixel_values=inp["pixel_values"], output_hidden_states=True)
        hid = out.decoder_hidden_states[-1].float(); pcls = out.logits.float().argmax(-1); pb = out.pred_boxes.float()
        pxy = torch.stack([pb[..., 0] - pb[..., 2] / 2, pb[..., 1] - pb[..., 3] / 2,
                           pb[..., 0] + pb[..., 2] / 2, pb[..., 1] + pb[..., 3] / 2], -1)
        for b, (r, im) in enumerate(zip(batch, ims)):
            W, H = im.size
            g = torch.tensor(r["boxes"], dtype=torch.float32, device=dev)
            gxy = g[:, :4] / torch.tensor([W, H, W, H], device=dev); gc = g[:, 4].long()
            iou = box_iou(gxy, pxy[b])
            ok = (iou >= 0.5) & (pcls[b][None, :] == gc[:, None])
            n_gt += len(gc)
            for i in torch.nonzero(ok.any(1)).flatten().tolist():
                q = int(torch.argmax(iou[i] * ok[i].float())); c = int(gc[i]); v = hid[b, q].cpu().numpy()
                seen[c] += 1; n_kept += 1
                if len(emb[c]) < cap:                      # reservoir sampling: mẫu đều trên toàn bộ ảnh
                    emb[c].append(v)
                else:
                    j = rng.integers(0, seen[c])
                    if j < cap:
                        emb[c][j] = v
        if (k // 16) % 500 == 0:
            print(f"    phần {part + 1}: {k + len(batch)}/{len(items)} ảnh | GT khớp được {n_kept}/{n_gt}", flush=True)
    np.savez(part_path(part), n_gt=n_gt, n_kept=n_kept,
             **{f"c{c}": np.asarray(emb[c], np.float32).reshape(-1, 256) for c in range(10)},
             **{f"seen{c}": seen[c] for c in range(10)})
    print(f"[V] Phần {part + 1}/{nparts}: {n_kept}/{n_gt} GT khớp được", flush=True)

elif MODE == "merge":
    from sklearn.decomposition import PCA
    from sklearn.covariance import LedoitWolf
    nparts = int(sys.argv[3])
    parts = []
    for i in range(nparts):
        with np.load(part_path(i)) as z:                        # đọc hết vào RAM rồi đóng file
            parts.append({k: z[k] for k in z.files})
    emb = {c: np.concatenate([p[f"c{c}"] for p in parts]) for c in range(10)}
    seen = {c: int(sum(p[f"seen{c}"] for p in parts)) for c in range(10)}
    n_gt = int(sum(p["n_gt"] for p in parts)); n_kept = int(sum(p["n_kept"] for p in parts))
    counts = {BDD_CLASSES[c]: len(emb[c]) for c in range(10)}
    print(f"[*] GT khớp được (IoU>=0.5, đúng lớp): {n_kept}/{n_gt} = {n_kept / max(n_gt, 1):.3f}")
    print(f"[*] Vector dùng để tính (sau giới hạn {CAP_PER_CLASS}/lớp): {counts}")
    X = np.vstack([emb[c] for c in range(10) if len(emb[c])])
    MIN_VEC = int(os.environ.get("LATENT_MIN_VECTORS", 256))
    assert len(X) >= MIN_VEC, f"Quá ít vector ({len(X)}): mô hình có thể chưa học được gì."
    # ---- Định dạng M29: PCA 64 + Σ_k^-1 theo lớp ----
    sub = X[np.random.default_rng(0).choice(len(X), min(len(X), 200000), replace=False)]
    pca = PCA(n_components=64, random_state=42).fit(sub)
    centroids, inv_covs, shrunk = {}, {}, []
    for c in range(10):
        if not len(emb[c]):
            continue
        Z = pca.transform(emb[c])
        centroids[c] = torch.tensor(Z.mean(0), dtype=torch.float32)
        if len(Z) >= 3 * 64:
            cov = np.cov(Z, rowvar=False)
        else:
            cov = LedoitWolf().fit(Z).covariance_ if len(Z) > 1 else np.eye(64)
            shrunk.append(BDD_CLASSES[c])
        inv_covs[c] = torch.tensor(np.linalg.inv(cov + JITTER * np.eye(64)), dtype=torch.float32)
    m29 = {"pca": pca, "centroids": centroids, "inv_covs": inv_covs, "class_names": BDD_CLASSES, "d_in": 256,
           "d_out": 64, "jitter_eps": JITTER, "shrunk_classes": shrunk, "n_vectors": counts,
           "n_matched_total": {BDD_CLASSES[c]: seen[c] for c in range(10)},
           "source": "toàn bộ BDD train, Max-IoU (IoU>=0.5, đúng lớp), decoder_hidden_states[-1]", "expert_id": EXPERT_ID}
    torch.save(m29, os.path.join(WORK, f"expert_{EXPERT_ID}_centroids.pt"))
    if EXPERT_ID == 1:
        torch.save(m29, os.path.join(WORK, "centroids.pt"))   # tên M29 dùng cho base model
    print(f"[V] Đã lưu expert_{EXPERT_ID}_centroids.pt" + (" và centroids.pt" if EXPERT_ID == 1 else "") +
          (f" | lớp dùng hiệp phương sai co rút: {shrunk}" if shrunk else ""))
    # ---- Cho H2: tâm 256D + hiệp phương sai gộp Ledoit-Wolf + chuẩn hóa robust ----
    means = np.zeros((10, 256), np.float32); within = []
    for c in range(10):
        if len(emb[c]):
            means[c] = emb[c].mean(0); within.append(emb[c] - means[c])
    W_ = np.vstack(within)
    W_ = W_[np.random.default_rng(1).choice(len(W_), min(len(W_), 200000), replace=False)]
    lw = LedoitWolf().fit(W_, ); P = lw.precision_.astype(np.float32)
    D = np.concatenate([np.sqrt(np.einsum("ij,jk,ik->i", emb[c] - means[c], P, emb[c] - means[c]))
                        for c in range(10) if len(emb[c])])
    med = float(np.median(D)); mad = float(np.median(np.abs(D - med)) * 1.4826)
    torch.save({"class_means": torch.tensor(means), "precision": torch.tensor(P), "shrinkage": torch.tensor(lw.shrinkage_),
                "tp_median_D": torch.tensor(med), "tp_mad_D": torch.tensor(mad),
                "n_vectors": torch.tensor([len(emb[c]) for c in range(10)])},
               os.path.join(WORK, f"expert_{EXPERT_ID}_latent_stats.pt"))
    for i in range(nparts):
        os.remove(part_path(i))                                 # M29: không giữ vector thô
    print(f"[V] Đã lưu expert_{EXPERT_ID}_latent_stats.pt | co rút Ledoit-Wolf {lw.shrinkage_:.3f} | "
          f"Mahalanobis TP: trung vị {med:.2f}, MAD {mad:.2f}")


In [ ]:
# Trích vector ẩn trên TOÀN BỘ tập train: mỗi GPU xử lý một phần song song, rồi gộp lại
import torch
n_part = max(1, torch.cuda.device_count())
cmd = " & ".join(f"python latent_stats.py extract {EXPERT_ID} {i} {n_part}" for i in range(n_part)) + " & wait"
!{cmd}
!python latent_stats.py merge {EXPERT_ID} {n_part}


In [ ]:
# ==============================================================================
# TÓM TẮT KẾT QUẢ
# ==============================================================================
import json, os
m = json.load(open(f"/kaggle/working/expert_{EXPERT_ID}_manifest.json"))
print(json.dumps({k: m[k] for k in ("expert_id", "seed", "steps_done", "epochs_done", "effective_batch",
                                    "train_filtered", "weights_sha256")}, indent=2))
for f in sorted(os.listdir("/kaggle/working")):
    p = os.path.join("/kaggle/working", f)
    print("    ->", f + ("/" if os.path.isdir(p) else f"  ({os.path.getsize(p) / 2**20:.1f} MB)"))
